In [ ]:
"""
contrastive_temporal_cae.py: Contrastive Temporal CAE for Vibration-Based Anomaly Detection


Extends the standard Temporal CAE (temporal_cae.py) by adding a SimCLR-style
self-supervised contrastive objective to the training loss.

How the contrastive term works
------------------------------
During each training step:
  1. For every sample x in the batch, a positive pair x+ is created by adding
     small Gaussian noise: x+ = x + ε,  ε ~ N(0, AUG_NOISE_SCALE²).
  2. Both x and x+ are forwarded through the encoder to obtain latent vectors
     z (anchor) and z+ (positive).
  3. An alignment loss pulls z and z+ together in the normalised latent space,
     encouraging the network to learn noise-invariant representations of normal
     operating conditions.
  4. The total training loss is:
         L = L_recon(x, x̂) + λ · L_contrastive(z, z+)

At inference time only reconstruction error is used as the anomaly score,
exactly as in the standard TCAE.

Dataset compatibility
---------------------
Identical to temporal_cae.py — controlled via DATASET_PAIRS:

    • HUMS2023  –  4095-point signals  (pad_len = 1, applied internally)
    • IMS and XJTUSY and XJTUSY       –  4096-point signals  (pad_len = 0, no padding)

Usage
-----
    python contrastive_temporal_cae.py

Requirements
------------
    torch, numpy, scipy, scikit-image, scikit-learn
    (see requirements.txt)
       
"""
%reset -f
import os
import json
import glob
import time
import re
import csv
import random
from typing import List, Tuple

import numpy as np
from numpy.lib.format import open_memmap
from scipy.io import loadmat
from skimage.filters import threshold_otsu
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader



#CONFIGURATION


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


DATASET_PAIRS: List[Tuple[str, str, str, int, str]] = [
    # --- HUMS2023 (Sensor 2) — 4095-point signals ---
    ("./data/training_until20",
     "./data/testAIRL_from21",
     "HUMS_RF2",
        4095,    # signal length; model pads 4095 → 4096 internally
        "xr",   # variable name inside each .mat file
    ),
    # --- IMS and XJTUSY bearing dataset — 4096-point signals ---
    # (
    #     "/path/to/IMS or XJTUSY/Training",
    #     "/path/to/IMS or XJTUSY/Testing",
    #     "IMS and XJTUSY",
    #     4096,   # signal length; no padding needed (4096 = 4^6)
    #     "ims or xjtusy",
    # ),
]

OUTPUT_DIR = "./contrastive_tcae_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)


BOTTLENECK_CH  = 64        # Encoder bottleneck channels
AE_DROPOUT     = 0.0
AE_LR          = 1e-3
AE_WEIGHT_DECAY= 1e-5
AE_EPOCHS      = 300
AE_BATCH       = 128
AE_PATIENCE    = 10        # Early-stopping patience (validation epochs)
AE_VAL_SPLIT   = 0.10
AE_MAX_SAMPLES = None      # Cap training set size per run; None = use all
AE_USE_L1_LOSS = False     # False → MSE loss, True → L1 loss for reconstruction
GRAD_CLIP_NORM = 1.0


# Contrastive learning hyper-parameters

LAMBDA_CONTRASTIVE = 0.1   # Weight λ of the contrastive loss term
CONTRASTIVE_TEMP   = 0.5   # Temperature τ in the cosine-similarity alignment loss
AUG_NOISE_SCALE    = 0.05  # σ of the Gaussian noise used to create positive pairs


N_RUNS = 10
SEEDS  = list(range(N_RUNS))


USE_SCALER  = False   # True → per-feature StandardScaler; False → global z-score
REMOVE_DC   = False   # Subtract per-sample mean before normalisation
SAVE_MODELS = True    # Persist best-epoch weights for each run


USE_MEAN   = True    # \
USE_MEDIAN = False   # > Enable exactly one of these three
USE_VOTE   = False   # /
ZSCORE_PER_RUN_BEFORE_ENSEMBLE = True
LOWER_TAIL = False   # True if anomalies have lower scores than normal
EPS = 1e-12




def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_folder_mat_1d(
    folder: str,
    key: str,
    expected_len: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Load every .mat file in *folder*, extract the 1-D array stored under *key*,
    and return a (N, expected_len) float32 matrix together with the file paths.
    """
    files = sorted(glob.glob(os.path.join(folder, "*.mat")))
    X: List[np.ndarray] = []
    F: List[str] = []

    for fp in files:
        try:
            mat = loadmat(fp)
            arr = mat.get(key)

            if arr is None:
                for v in mat.values():
                    if isinstance(v, np.ndarray) and v.size == expected_len:
                        arr = v
                        break

            if arr is None:
                continue

            x = np.asarray(arr, dtype=np.float32).reshape(-1)
            if len(x) != expected_len:
                continue

            X.append(x)
            F.append(fp)
        except Exception as exc:  # noqa: BLE001
            print(f"  [warning] Could not load {fp}: {exc}")

    if not X:
        raise RuntimeError(f"No usable .mat files found in: {folder}")

    data = np.vstack(X)
    data = np.nan_to_num(data, copy=False)
    return data, np.array(F, dtype=object)


def standardize_train_test(
    Xtr_raw: np.ndarray,
    Xte_raw: np.ndarray,
    mode: str = "train",
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Normalise train and test sets.

    When *mode* == 'train' the normalisation statistics are computed on the
    training set and applied to both splits (no data leakage).
    Returns (Xtr, Xte, scaler_mean, scaler_scale).
    """
    Xtr = Xtr_raw.copy()
    Xte = Xte_raw.copy()

    if REMOVE_DC:
        Xtr = Xtr - Xtr.mean(axis=1, keepdIMS and XJTUSY=True)
        Xte = Xte - Xte.mean(axis=1, keepdIMS and XJTUSY=True)

    if mode.lower() == "train":
        if USE_SCALER:
            sc = StandardScaler().fit(Xtr.reshape(-1, 1))
            Xtr = sc.transform(Xtr.reshape(-1, 1)).reshape(Xtr.shape)
            Xte = sc.transform(Xte.reshape(-1, 1)).reshape(Xte.shape)
            return Xtr, Xte, sc.mean_.astype(np.float32), sc.scale_.astype(np.float32)
        else:
            mu  = Xtr.flatten().mean()
            std = Xtr.flatten().std()
            Xtr = (Xtr - mu) / (std + 1e-9)
            Xte = (Xte - mu) / (std + 1e-9)
            return (
                Xtr, Xte,
                np.full(Xtr.shape[1], mu,  dtype=np.float32),
                np.full(Xtr.shape[1], std, dtype=np.float32),
            )

    return (
        Xtr, Xte,
        np.zeros(Xtr.shape[1], dtype=np.float32),
        np.ones(Xtr.shape[1],  dtype=np.float32),
    )


def subsample_rows(X: np.ndarray, max_samples: int | None, seed: int) -> np.ndarray:
    """Randomly subsample rows of *X* if it exceeds *max_samples*."""
    if max_samples is None or max_samples >= len(X):
        return X
    idx = np.random.RandomState(seed).choice(len(X), size=max_samples, replace=False)
    return X[idx]


def ensure_channel_dir(tag: str, stamp: str) -> str:
    """Create and return a per-experiment output directory."""
    path = os.path.join(OUTPUT_DIR, f"{tag}_{stamp}")
    os.makedirs(path, exist_ok=True)
    return path


def write_json_safely(path: str, obj: object) -> None:
    """Write *obj* to *path* atomically (write-then-rename)."""
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2)
    os.replace(tmp, path)



#CONTRASTIVE LOSS


def contrastive_loss(
    z_anchor: torch.Tensor,
    z_positive: torch.Tensor,
    temperature: float = CONTRASTIVE_TEMP,
) -> torch.Tensor:
    """
    Pairwise alignment contrastive loss for anomaly-detection training.

    Maximises cosine similarity between the anchor embedding *z_anchor* and its
    noise-augmented positive counterpart *z_positive*, divided by the softmax
    temperature.  The formulation is:

        L = -log( mean( exp( sim(z, z+) / τ ) ) )

    where sim denotes cosine similarity after L2 normalisation.

    This is a stable, lightweight alternative to full NT-Xent for the
    small-to-medium batch sizes typical in bearing fault detection.  Full
    NT-Xent (which treats all other batch items as negatives) requires careful
    tuning of batch size and temperature; the alignment formulation used here
    converges reliably without that overhead.

    Parameters
    ----------
    z_anchor   : (batch, latent_dim) — encoder output for original samples
    z_positive : (batch, latent_dim) — encoder output for augmented samples
    temperature : float — controls sharpness of the similarity distribution

    Returns
    -------
    Scalar loss tensor.
    """
    z_anchor   = nn.functional.normalize(z_anchor,   dim=1)
    z_positive = nn.functional.normalize(z_positive, dim=1)

    # Element-wise dot product → cosine similarity, scaled by temperature
    cos_sim = torch.sum(z_anchor * z_positive, dim=1) / temperature

    # Negative log of the mean exponentiated similarity
    return -torch.log(torch.exp(cos_sim).mean())



#MODEL

class ContrastiveTemporalCAE(nn.Module):
    """
    Temporal Convolutional Autoencoder with a contrastive-learning extension.

    The architecture is identical to TemporalCAE in temporal_cae.py.  The
    difference is the additional ``return_latent`` flag in :meth:`forward`:
    when set to True the method also returns the flattened latent vector so that
    the contrastive loss can be computed in the training loop.

    Parameters
    ----------
    in_dim : int
        Length of each input signal (4095 for HUMS, 4096 for IMS and XJTUSY).
    bottleneck_ch : int
        Number of channels in the encoder bottleneck layer.
    dropout : float
        Dropout probability applied after each intermediate layer.
    """

    _PADDED_LEN: int = 4096

    def __init__(
        self,
        in_dim: int = 4095,
        bottleneck_ch: int = 64,
        dropout: float = 0.0,
    ) -> None:
        super().__init__()
        self.in_dim    = in_dim
        self.pad_len   = self._PADDED_LEN - in_dim

        # --- Encoder ---
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=4, padding=4),
            nn.BatchNorm1d(16), nn.ReLU(inplace=True), nn.Dropout(dropout),

            nn.Conv1d(16, 32, kernel_size=9, stride=4, padding=4),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(dropout),

            nn.Conv1d(32, bottleneck_ch, kernel_size=9, stride=4, padding=4),
            nn.ReLU(inplace=True),
        )

        # --- Decoder ---
        # output_padding=3 is required to restore the spatial dimension after
        # each stride-4 transposed convolution (see temporal_cae.py for derivation).
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(bottleneck_ch, 32, kernel_size=9, stride=4, padding=4, output_padding=3),
            nn.BatchNorm1d(32), nn.ReLU(inplace=True), nn.Dropout(dropout),

            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=4, padding=4, output_padding=3),
            nn.BatchNorm1d(16), nn.ReLU(inplace=True), nn.Dropout(dropout),

            nn.ConvTranspose1d(16, 1, kernel_size=9, stride=4, padding=4, output_padding=3),
        )

    def forward(
        self,
        x: torch.Tensor,
        return_latent: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, torch.Tensor]:
        
        x = x.unsqueeze(1)  # → (batch, 1, in_dim)

        if self.pad_len > 0:
            x = nn.functional.pad(x, (0, self.pad_len), mode="replicate")

        z     = self.encoder(x)
        x_hat = self.decoder(z)[:, :, : self.in_dim].squeeze(1)  # → (batch, in_dim)

        if return_latent:
            return x_hat, z.view(z.size(0), -1)  # flatten (batch, ch, t) → (batch, ch*t)

        return x_hat


def validate_model_shapes(in_dim: int = 4095, bottleneck_ch: int = 64) -> None:
    """Sanity-check that the model produces the correct output and latent shapes."""
    print("\n[Validation] Checking model shapes ...")
    model = ContrastiveTemporalCAE(in_dim=in_dim, bottleneck_ch=bottleneck_ch).to(DEVICE)
    model.eval()
    x = torch.randn(2, in_dim, device=DEVICE)

    with torch.no_grad():
        y, z = model(x, return_latent=True)
        assert x.shape == y.shape, f"FATAL: output {y.shape} != input {x.shape}"
        print(f"  ✓ End-to-end shape: {tuple(x.shape)} → {tuple(y.shape)}")
        print(f"  ✓ Latent shape (flattened): {tuple(z.shape)}")

    print("[Validation] All checks passed.\n")


def print_model_summary(model: nn.Module, in_dim: int) -> None:
    """Print the model architecture and parameter count."""
    print(model)
    n_total = sum(p.numel() for p in model.parameters())
    print(f"\nTotal parameters: {n_total:,}")
    model.eval()
    with torch.no_grad():
        y, z = model(torch.randn(2, in_dim, device=DEVICE), return_latent=True)
    print(f"Input  shape: (batch, {in_dim})")
    print(f"Latent shape: {tuple(z.shape)}")
    print(f"Output shape: {tuple(y.shape)}\n")



# TRAINING


@torch.no_grad()
def reconstruction_scores(
    model: nn.Module,
    X: np.ndarray,
    use_l1: bool,
) -> np.ndarray:
    """
    Compute per-sample reconstruction errors (MSE or L1) for all rows of *X*.

    Only reconstruction error is used as the anomaly score at inference time;
    the contrastive term is training-only.
    """
    model.eval()
    errors: List[np.ndarray] = []

    for i in range(0, len(X), 2048):
        xb  = torch.from_numpy(X[i : i + 2048]).to(DEVICE)
        xh  = model(xb)
        err = (xb - xh).abs().mean(dim=1) if use_l1 else ((xb - xh) ** 2).mean(dim=1)
        errors.append(err.cpu().numpy().astype(np.float32))

    return np.concatenate(errors, axis=0)


def train_one_run(Xtr: np.ndarray, seed: int, in_dim: int) -> nn.Module:
    set_seed(seed)
    Xfit = subsample_rows(Xtr, AE_MAX_SAMPLES, seed)

    n     = len(Xfit)
    n_val = max(1, int(AE_VAL_SPLIT * n))
    perm  = np.random.permutation(n)
    val_idx, tr_idx = perm[:n_val], perm[n_val:]

    Xtr_tensor = torch.from_numpy(Xfit[tr_idx])
    Xva_tensor = torch.from_numpy(Xfit[val_idx]).to(DEVICE)

    loader = DataLoader(
        TensorDataset(Xtr_tensor),
        batch_size=AE_BATCH,
        shuffle=True,
        drop_last=False,
    )

    model       = ContrastiveTemporalCAE(in_dim=in_dim, bottleneck_ch=BOTTLENECK_CH, dropout=AE_DROPOUT).to(DEVICE)
    opt         = torch.optim.Adam(model.parameters(), lr=AE_LR, weight_decay=AE_WEIGHT_DECAY)
    recon_crit  = nn.L1Loss() if AE_USE_L1_LOSS else nn.MSELoss()

    best_val         = float("inf")
    best_state       = None
    patience_counter = 0

    for _ in range(1, AE_EPOCHS + 1):
        # --- Training pass (hybrid loss) ---
        model.train()
        for (xb,) in loader:
            xb    = xb.to(DEVICE)
            # Create positive pair: add Gaussian noise
            xb_aug = xb + torch.randn_like(xb) * AUG_NOISE_SCALE

            opt.zero_grad(set_to_none=True)

            xh, z      = model(xb,     return_latent=True)   # anchor
            _,  z_aug  = model(xb_aug, return_latent=True)   # positive

            l_recon   = recon_crit(xh, xb)
            l_contra  = contrastive_loss(z, z_aug, temperature=CONTRASTIVE_TEMP)
            loss      = l_recon + LAMBDA_CONTRASTIVE * l_contra

            loss.backward()
            opt.step()

        # --- Validation pass (reconstruction only, no augmentation) ---
        model.eval()
        with torch.no_grad():
            val_loss = recon_crit(model(Xva_tensor), Xva_tensor).item()

        if val_loss < best_val - 1e-6:
            best_val         = val_loss
            best_state       = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= AE_PATIENCE:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    model.eval()
    return model


def open_or_init_memmaps(
    chan_dir: str,
    n_runs: int,
    n_train: int,
    n_test: int,
    suffix: str = "",
) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Open (or create) memory-mapped arrays for incremental score accumulation.

    Allows a crashed experiment to be resumed without re-running completed seeds.
    Returns (mm_train, mm_test, first_incomplete_run_index).
    """
    tr_path = os.path.join(chan_dir, f"train_scores_runs{suffix}.npy")
    te_path = os.path.join(chan_dir, f"test_scores_runs{suffix}.npy")

    if os.path.exists(tr_path) and os.path.exists(te_path):
        mm_tr = open_memmap(tr_path, mode="r+")
        mm_te = open_memmap(te_path, mode="r+")
        start = next(
            (r for r in range(mm_tr.shape[0]) if np.all(np.isnan(mm_tr[r]))),
            mm_tr.shape[0],
        )
    else:
        mm_tr = open_memmap(tr_path, mode="w+", dtype="float32", shape=(n_runs, n_train))
        mm_te = open_memmap(te_path, mode="w+", dtype="float32", shape=(n_runs, n_test))
        mm_tr[:] = np.nan
        mm_te[:] = np.nan
        mm_tr.flush()
        mm_te.flush()
        start = 0

    return mm_tr, mm_te, start


#THRESHOLDING AND EVALUATION

def _natural_key(name: str) -> tuple:
    """Sort key that orders filenames numerically."""
    return tuple(
        int(p) if p.isdigit() else p
        for p in re.split(r"(\d+)", name)
    )


def _check_ensemble_mode() -> str:
 
    n_active = sum([USE_MEAN, USE_MEDIAN, USE_VOTE])
    if n_active != 1:
        raise ValueError(
            f"Exactly one of USE_MEAN / USE_MEDIAN / USE_VOTE must be True "
            f"(currently {n_active} are True)."
        )
    if USE_MEAN:   return "mean"
    if USE_MEDIAN: return "median"
    return "vote"


def ensemble_scores(
    train_runs: np.ndarray,
    test_runs:  np.ndarray,
    zscore_per_run: bool = True,
) -> Tuple[np.ndarray, np.ndarray, str]:
    tr = train_runs.astype(float).copy()
    te = test_runs.astype(float).copy()

    if zscore_per_run:
        for r in range(tr.shape[0]):
            mu, sd = train_runs[r].mean(), train_runs[r].std() + 1e-12
            tr[r]  = (train_runs[r] - mu) / sd
            te[r]  = (test_runs[r]  - mu) / sd

    mode = _check_ensemble_mode()

    if mode == "mean":
        return tr.mean(axis=0), te.mean(axis=0), mode
    if mode == "median":
        return np.median(tr, axis=0), np.median(te, axis=0), mode

    taus     = np.percentile(tr, 99, axis=1)
    votes_tr = (tr > taus[:, None]).astype(int)
    votes_te = (te > taus[:, None]).astype(int)
    return votes_tr.mean(axis=0), votes_te.mean(axis=0), mode


def build_thresholds(train_scores: np.ndarray) -> dict:
    tr    = np.asarray(train_scores, float)
    mu    = float(tr.mean())
    sigma = float(tr.std() + 1e-12)

    thresholds = {
        "mean_plus_2std": mu + 2 * sigma,
        "mean_plus_3std": mu + 3 * sigma,
        "p99":            float(np.percentile(tr, 99)),
        "max":            float(tr.max()),
    }

    try:
        km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(tr.reshape(-1, 1))
        c0, c1 = np.sort(km.cluster_centers_.flatten())
        thresholds["kmeans_mid"] = float(0.5 * (c0 + c1))
    except Exception:  # noqa: BLE001
        pass

    try:
        thresholds["otsu"] = float(threshold_otsu(tr))
    except Exception:  # noqa: BLE001
        pass

    return thresholds


def natural_order_indices(file_list: np.ndarray) -> np.ndarray:
    names = [os.path.basename(str(f)) for f in file_list]
    return np.array(
        sorted(range(len(names)), key=lambda i: _natural_key(names[i])),
        dtype=int,
    )



#MAIN EXECUTION PIPELINE


if __name__ == "__main__":

    print(f"Device: {DEVICE}\n")

    first_len = DATASET_PAIRS[0][3]
    validate_model_shapes(in_dim=first_len, bottleneck_ch=BOTTLENECK_CH)

    print("PHASE 1: Training Contrastive Temporal CAE")
    print(f"         λ={LAMBDA_CONTRASTIVE}  τ={CONTRASTIVE_TEMP}  "
          f"noise_σ={AUG_NOISE_SCALE}")


    stamp              = "ContrastiveTCAE"
    generated_npz_list: List[str] = []

    for train_dir, test_dir, tag, expected_len, mat_key in DATASET_PAIRS:
        print(f"\n--- Dataset: {tag}  |  signal length: {expected_len} ---")

        Xtr_raw, Ftr = load_folder_mat_1d(train_dir, mat_key, expected_len)
        Xte_raw, Fte = load_folder_mat_1d(test_dir,  mat_key, expected_len)

        Xtr, Xte, scaler_mean, _ = standardize_train_test(Xtr_raw, Xte_raw, mode="train")

        chan_dir = ensure_channel_dir(tag, stamp)

        write_json_safely(
            os.path.join(chan_dir, "meta.json"),
            dict(
                tag=tag, stamp=stamp, ae_type="ContrastiveTemporalCAE",
                signal_len=expected_len, mat_key=mat_key,
                n_train=int(Xtr.shape[0]),
                lambda_contrastive=LAMBDA_CONTRASTIVE,
                contrastive_temp=CONTRASTIVE_TEMP,
                aug_noise_scale=AUG_NOISE_SCALE,
                scaler_mean=scaler_mean.tolist(),
            ),
        )

        tmp = ContrastiveTemporalCAE(expected_len, bottleneck_ch=BOTTLENECK_CH).to(DEVICE)
        print_model_summary(tmp, in_dim=expected_len)
        del tmp

        mm_tr, mm_te, start_run = open_or_init_memmaps(
            chan_dir, N_RUNS, Xtr.shape[0], Xte.shape[0],
            suffix=f"_{tag}",
        )

        prog_path = os.path.join(chan_dir, "progress.json")
        if os.path.exists(prog_path):
            prog      = json.load(open(prog_path))
            start_run = max(start_run, int(prog.get("last_completed_run", -1)) + 1)

        for r in range(start_run, N_RUNS):
            seed = SEEDS[r]
            t0   = time.time()

            model = train_one_run(Xtr, seed, expected_len)

            if SAVE_MODELS:
                torch.save(
                    model.state_dict(),
                    os.path.join(chan_dir, f"run{r:02d}_seed{seed}_best.pt"),
                )

            mm_tr[r] = reconstruction_scores(model, Xtr, AE_USE_L1_LOSS)
            mm_te[r] = reconstruction_scores(model, Xte, AE_USE_L1_LOSS)
            mm_tr.flush()
            mm_te.flush()

            elapsed = time.time() - t0
            print(f"  Run {r + 1:>2}/{N_RUNS}  seed={seed}  ({elapsed:.1f}s)")
            write_json_safely(prog_path, dict(last_completed_run=r, seeds=SEEDS))

        final_npz = os.path.join(chan_dir, f"{tag}_{stamp}_runs.npz")
        np.savez_compressed(
            final_npz,
            train_scores_runs=np.array(mm_tr),
            test_scores_runs =np.array(mm_te),
            train_files=Ftr,
            test_files =Fte,
        )
        print(f"  Saved: {final_npz}")
        generated_npz_list.append(final_npz)

    print("\n" + "=" * 60)
    print("PHASE 2: Thresholding and Anomaly Detection Reports")


    for npz_path in generated_npz_list:
        if not os.path.isfile(npz_path):
            continue

        data     = np.load(npz_path, allow_pickle=True)
        tr_runs  = data["train_scores_runs"].astype(float)
        te_runs  = data["test_scores_runs"].astype(float)
        tr_files = data["train_files"].astype(object)
        te_files = data["test_files"].astype(object)

        tag = os.path.basename(npz_path).replace(".npz", "")

        tr_ens, te_ens, mode_name = ensemble_scores(
            tr_runs, te_runs, zscore_per_run=ZSCORE_PER_RUN_BEFORE_ENSEMBLE
        )
        thresholds = build_thresholds(tr_ens)
        nat_idx    = natural_order_indices(te_files)

        counts_csv   = os.path.join(OUTPUT_DIR, f"{tag}_{mode_name}_counts.csv")
        earliest_csv = os.path.join(OUTPUT_DIR, f"{tag}_{mode_name}_earliest.csv")

        with open(counts_csv, "w", newline="") as fc, \
             open(earliest_csv, "w", newline="") as fe:

            wc = csv.writer(fc)
            we = csv.writer(fe)
            wc.writerow(["threshold_name", "tau", "n_anomalous", "n_normal"])
            we.writerow(["threshold_name", "tau", "earliest_file", "anom_after_first"])

            for tname in sorted(thresholds):
                tau   = thresholds[tname]
                flags = (te_ens <= tau + EPS) if LOWER_TAIL else (te_ens >= tau - EPS)
                n_anom = int(flags.sum())

                wc.writerow([tname, f"{tau:.6f}", n_anom, int((~flags).sum())])

                if n_anom > 0:
                    flags_nat = flags[nat_idx]
                    first_idx = int(np.argmax(flags_nat))
                    if flags_nat[first_idx]:
                        fname      = os.path.basename(str(te_files[nat_idx[first_idx]]))
                        anom_after = int(flags_nat[first_idx:].sum())
                        we.writerow([tname, f"{tau:.6f}", fname, anom_after])
                    else:
                        we.writerow([tname, f"{tau:.6f}", "None", 0])
                else:
                    we.writerow([tname, f"{tau:.6f}", "None", 0])

        print(f"  Reports written for {tag}")

    print("\n[DONE] Contrastive Temporal CAE experiment complete.")
